# Split daily chat into enriched topic chunks (LLM)

Reads each `data/chats/<year>-<month>-<day>/raw_chat.json`, asks Gemini to group messages by topic, and writes one file per topic:

```
data/chats/23-6-27/
  raw_chat.json
  chunk_1.json
  chunk_2.json
```

Each `chunk_N.json`:

```json
{
  "date": "2023-06-27",
  "topic_summary": "...",
  "topic_scores": {
    "valence": 0.58,
    "arousal": 0.47,
    "stress": 0.41,
    "engagement": 0.93,
    "impact": 0.5
  },
  "keywords": ["keyword1", "keyword2"],
  "messages": [ ... ]
}
```

Requires `GEMINI_API_KEY` in `backend/.env`.

In [ ]:
from pathlib import Path

from chunk_by_topic import load_gemini_config, split_day_by_topic

root = Path("..").resolve()
chats_root = root / "data" / "chats"
env_path = root / "backend" / ".env"

api_key, model = load_gemini_config(env_path)
print(f"Gemini model: {model}")
print(f"Day folders: {len(list(chats_root.glob('*/raw_chat.json')))}")

In [ ]:
import json

# Example: one small day
example_day = chats_root / "23-6-27"
raw_chat_path = example_day / "raw_chat.json"

written = split_day_by_topic(raw_chat_path, env_path=env_path)
print(f"Wrote {len(written)} topic file(s):")
for path in written:
    print(f"  {path.name}")

if written:
    json.loads(written[0].read_text(encoding="utf-8"))

In [ ]:
# Example: one busier day (multiple topics expected)
busy_day = chats_root / "23-7-3"
busy_written = split_day_by_topic(busy_day / "raw_chat.json", env_path=env_path)
print(f"Wrote {len(busy_written)} topic file(s) for {busy_day.name}")
for path in busy_written:
    payload = json.loads(path.read_text(encoding="utf-8"))
    print(f"- {path.name}: {payload['topic_summary'][:80]}... ({len(payload['messages'])} msgs)")

In [ ]:
# Optional: run for every day (costs one LLM call per day)
# from chunk_by_topic import split_all_days
# all_written = split_all_days(chats_root, env_path=env_path)
# print(f"Wrote {len(all_written)} chunk files across all days")